In [2]:
!pip install hazm
!pip install tensorflow_gpu==2.0

     |████████████████████████████████| 316 kB 5.5 MB/s 
     |████████████████████████████████| 233 kB 41.4 MB/s 
     |████████████████████████████████| 1.4 MB 39.5 MB/s 
  Created wheel for nltk: filename=nltk-3.3-py3-none-any.whl size=1394486 sha256=ab041b0c201989bb5356e75d9d012334b895b336aa103449d1fb90d4908828a9
  Stored in directory: /root/.cache/pip/wheels/9b/fd/0c/d92302c876e5de87ebd7fc0979d82edb93e2d8d768bf71fac4
  Created wheel for libwapiti: filename=libwapiti-0.2.1-cp37-cp37m-linux_x86_64.whl size=154115 sha256=4c0a87974ff9056f3279af488fa140396c1b617d294c756cd160d4779d6b7734
  Stored in directory: /root/.cache/pip/wheels/ab/b2/5b/0fe4b8f5c0e65341e8ea7bb3f4a6ebabfe8b1ac31322392dbf
Successfully built nltk libwapiti
  Attempting uninstall: nltk
    Found existing installation: nltk 3.2.5
    Uninstalling nltk-3.2.5:
      Successfully uninstalled nltk-3.2.5
     |████████████████████████████████| 380.8 MB 38 kB/s 
     |████████████████████████████████| 50 kB 6.7 MB/s 
     |█

In [3]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Input, Embedding, Dropout, GlobalMaxPool1D, Conv1D, MaxPooling1D, Conv2D, MaxPooling2D, SpatialDropout1D
from tensorflow.keras.layers import LSTM, Bidirectional, GRU, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from hazm import *
 
import re

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset=pd.read_table('/content/drive/MyDrive/Poems/20078.persian_poem.txt')
dataset

,لله الحمد قبل کل کلام
0,به صفات الجلال و الاکرام
1,هر چه مفهوم عقل و ادراک است
2,ساحت قدس او از آن پاک است
3,به هوا و هوس در او نرسی
4,تا ز لا نگذری به هو نرسی
5,ای همه قدسیان قدوسی
6,گرد کوی تو در زمین بوسی!
7,پرتو روی توست از همه سو
8,همه را رو به توست از همه رو
9,قطع این ره به راه‌پیمایی


In [ ]:
sentence_train=np.asarray(dataset)
sentence_train

array([['به صفات الجلال و الاکرام'],
       ['هر چه مفهوم عقل و ادراک است'],
       ['ساحت قدس او از آن پاک است'],
       ['به هوا و هوس در او نرسی'],
       ['تا ز لا نگذری به هو نرسی'],
       ['ای همه قدسیان قدوسی'],
       ['گرد کوی تو در زمین بوسی!'],
       ['پرتو روی توست از همه سو'],
       ['همه را رو به توست از همه رو'],
       ['قطع این ره به راه\u200cپیمایی'],
       ['کی توان گر تو راه ننمایی ؟'],
       ['بنما ره! که طالب راهیم'],
       ['ره به سوی تو از تو می\u200cخواهیم'],
       ['احدی، لیک مرجع اعداد'],
       ['واحدی، لیک مجمع اضداد'],
       ['اولی و تو را بدایت نی'],
       ['آخری و تو را نهایت نی'],
       ['ذات تو در سرادقات جلال'],
       ['از ازل تا ابد به یک منوال'],
       ['بر تو کس نیست آمر و ناهی'],
       ['همه آن می\u200cکنی که می\u200cخواهی'],
       ['ای جهانی به کام، از در تو!'],
       ['کام خواهم نه دام از در تو'],
       ['به جوار خودم رهی بنمای!'],
       ['در حریم دلم دری بگشای!'],
       ['غایب از من، مرا حضوری بخش!'],
       ['به سروری رسان و 

In [9]:
datasett=pd.read_table('/content/drive/MyDrive/Stopwords/Stopwords.txt', delimiter='\t')
stops=list(datasett['value'])
#sstop=list(np.asarray(datasett))
stops
#datasett
stop_words = set((stops))
stop_words

{'آمد',
 'آن',
 'آید',
 'از',
 'است',
 'اندر',
 'او',
 'اگر',
 'ای',
 'این',
 'با',
 'باشد',
 'بر',
 'به',
 'بود',
 'تا',
 'تو',
 'خود',
 'دارد',
 'در',
 'را',
 'ز',
 'سر',
 'سوی',
 'شد',
 'شود',
 'ما',
 'مر',
 'نه',
 'نیست',
 'هر',
 'هم',
 'همه',
 'همی',
 'و',
 'پر',
 'پس',
 'پیش',
 'چنین',
 'چه',
 'چو',
 'چون',
 'کرد',
 'کز',
 'کن',
 'کند',
 'که',
 'کی',
 'گر',
 'یا'}

In [6]:
!pip install -q clean-text[gpl]
from cleantext import clean 

     |████████████████████████████████| 64 kB 1.8 MB/s 
     |████████████████████████████████| 170 kB 11.0 MB/s 
     |████████████████████████████████| 235 kB 40.0 MB/s 


In [43]:
#preprocessing


normalizer=Normalizer()
lemmatizer=Lemmatizer()
stemmer=Stemmer()
#turn a doc into clean token
def clean_data(doc):
  doc=normalizer.normalize(doc)
  tokenized=word_tokenize(doc)
  #print(tokenized)
  tokens=[]
  for token in tokenized:
    token=re.sub('[a-zA-Z0-9]+',"",token)
    token=re.sub("[.:,()/+]","",token)
    token=re.sub(r"\!+","!",token)
    token=re.sub(r"\؟+","؟",token)
    #token = re.sub(r"*\#*","",token)
    token = token.replace("#",'')

    token = token.replace("\u200c",' ')
   # token=re.sub("\u200c","",token)
    #token=re.sub("[۰۱۲۳۴۵۶۷۸۹]+"," ",token)
    #token = token.replace('[۰۱۲۳۴۵۶۷۸۹]+','')
    #token = token.replace('[a-zA-Z0-9]+','')
   
    doc=normalizer.normalize(doc)
    
    
    # removing wierd patterns
    wierd_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u'\U00010000-\U0010ffff'
        u"\u200d"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\u23cf"
        u"\u23e9"
        u"\u231a"
        u"\u3030"
        u"\ufe0f"
        u"\u2069"
        u"\u2066"
        # u"\u200c"
        u"\u2068"
        u"\u2067"
        "]+", flags=re.UNICODE)
    
    #token = wierd_pattern.sub(r'', token)
    cleanr = re.compile('<.*?>')
    token = re.sub(cleanr, '', token)
    # removing extra spaces, hashtags
    token = re.sub("[#+]", "", token)
    token = re.sub("$", "", token)
    token = re.sub("\s+", "", token)
     #if token.contain()
    
    tokens.append(token)
    #print(tokens)
  tokens=[w for w in tokens if not len(w)<=1]
  tokens=[w for w in tokens if not w.isdigit()]
  tokens=[lemmatizer.lemmatize(w) for w in tokens ]
  tokens=[w for w in tokens if w not in stop_words]
  #if token not in stop_words:
     # tokens.append(token)
  #tokens=[stemmer.stem(w) for w in tokens ]
  #print(tokens)
  tokens=' '.join(tokens)
  tokens = re.sub("[#]", "", tokens)
  #tokens = re.sub("[$]", "", tokens)
  return tokens

In [45]:
# Import Module
import os
  
# Folder Path
path = "/content/drive/MyDrive/Poems"
  
# Change the directory
os.chdir(path)
  
# Read text File
  
def read_text_file(file_path):
    with open(file_path, 'r') as f:
       # print(f.read())
      #tt=(f.read()).asarray()
       #print(f.read())
       newfile= clean_data(str(f.read()))
       #newfile=f.read()
       print(newfile)
       print('\n')
        #newfile=f.read()
       fileText=open('outputText.txt', 'a',encoding='utf-8')  
       fileText.write(newfile)
       fileText.close()
# iterate through all file
for file in os.listdir():
    # Check whether file is in text format or not
     if file.endswith(".persian_poem"):
        file_path = f"{path}/{file}"
  
        # call read text file function
        read_text_file(file_path)
       #print(file.encode("utf-8"))
     fileText=file.encode("utf-8")

رستخیز ناگهان وی رحمت بیمنتها آتش افروخته بیشه اندیشه امروز خندان آمدآ مفتاح زندان آمدآ مستمند آمدآ بخشش فضل خدا خورشید حاجب اومید واجب مطلب طالب منتها مبتدا سینه برخاسته اندیشه آراسته خویش حاجت خواسته خویشتن کرده روا روح بخش بیبدل وی لذت علم عمل باقی بهانهست دغل کاین علت آمدآ وان دوا زان دغل کژبین شده بیگنه کین شده گه مست حورالعین شده گه مست نان شوربا سکر بین هل عقل وین نقل بین هل نقل بهر نان بقل چندین نشاید ماجرا تدبیر صدرنگ افکنی روم زنگ افکنی میان جنگ افکنی فی اصطناع لا یری میمال پنهان گوش جان مینه بهانه کس جان رب خلصنی زن والله لاغست کیا خام بس مستعجل رفترو سو پای علم کاغذ بنه شکستشکن قلم ساقی درآمد الصلا


یوسف خوش نام خوش میروی بام درشکسته جام بردریده دام نور سور دولت منصور جوش بنه شور میشود انگور دلبر مقصود قبله معبود آتش زدزن عود نظاره دود یار عیار دام دل خمار پا وامکش کار بستان گرو دستار گل بمانده پای دل جان میدهم جای دل وز آتش سودا دل وای دل وای


شکل بین وان شیوه بین وان قد خد دست پا رنگ بین وان هنگ بین وان ماه بدر قبا سرو گفتگو چمن لاله گفتگو سمن شمع گفتگو لگن رقص گل صبا ع